In [1]:
!pip install transformers datasets nltk -q

import torch
from transformers import BlipProcessor, BlipForConditionalGeneration, TrainingArguments, Trainer
from PIL import Image
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

nltk.download('punkt')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:
model_name = "Salesforce/blip-image-captioning-base"

processor = BlipProcessor.from_pretrained(model_name)
model = BlipForConditionalGeneration.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BlipForConditionalGeneration(
  (vision_model): BlipVisionModel(
    (embeddings): BlipVisionEmbeddings(
      (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (encoder): BlipEncoder(
      (layers): ModuleList(
        (0-11): 12 x BlipEncoderLayer(
          (self_attn): BlipAttention(
            (dropout): Dropout(p=0.0, inplace=False)
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (projection): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): BlipMLP(
            (activation_fn): GELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        )
      )
    )
    (post_layernorm): LayerNorm((768,), eps=1e-0

In [3]:
import torch
import os
import pandas as pd
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from transformers import BitsAndBytesConfig

In [4]:
df = pd.read_csv("/kaggle/input/datasets/ankitusa/temporaldatasetwithoutcaption-2ankitnitb777/caption/caption.csv")

# df = df.head(1000)

images_dir = "/kaggle/input/datasets/ankitusa/temporaldatasetwithoutcaption-2ankitnitb777"

train_df, valid_df = train_test_split(df, test_size=0.2)
val_df, test_df = train_test_split(valid_df, test_size=0.5)
print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:",len(test_df))

Train size: 621
Validation size: 78
Test size: 78


In [5]:
from torch.utils.data import Dataset
from PIL import Image
import os

class FlickrDataset(Dataset):

    def __init__(self, dataframe, image_folder, processor):
        self.df = dataframe
        self.image_folder = image_folder
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_folder, row["image"])
        caption = row["caption"]

        image = Image.open(img_path).convert("RGB")

        encoding = self.processor(
            images=image,
            text=caption,
            padding="max_length",
            truncation=True,
            max_length=50,
            return_tensors="pt"
        )

        item = {
            "pixel_values": encoding["pixel_values"].squeeze(),
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": encoding["input_ids"].squeeze()
        }

        return item

In [6]:
train_dataset = FlickrDataset(train_df, images_dir, processor)
val_dataset = FlickrDataset(val_df, images_dir, processor)

In [7]:
sample = train_dataset[0]
print(sample.keys())

dict_keys(['pixel_values', 'input_ids', 'attention_mask', 'labels'])


In [8]:
train_dataset = FlickrDataset(train_df, images_dir, processor)

val_dataset = FlickrDataset(val_df, images_dir, processor)

print("Datasets created")

Datasets created


In [9]:
print(len(train_dataset))
print(len(val_dataset))

621
78


In [10]:
import torch

def custom_collator(batch):

    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    input_ids = torch.stack([item["input_ids"] for item in batch])
    attention_mask = torch.stack([item["attention_mask"] for item in batch])
    labels = torch.stack([item["labels"] for item in batch])

    return {
        "pixel_values": pixel_values,
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [11]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/blip-finetuned",

    per_device_train_batch_size=16,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    learning_rate=1e-5,
    num_train_epochs=20,

    # ✅ IMPORTANT IMPROVEMENTS
    lr_scheduler_type="cosine",
    warmup_steps=500,

    save_strategy="no",
    remove_unused_columns=False,
    report_to="none",
    fp16=True
)

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


In [ ]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/blip-finetuned-stage2",

    per_device_train_batch_size=16,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    learning_rate=5e-6,   # lower LR
    num_train_epochs=15,

    lr_scheduler_type="cosine",
    warmup_steps=200,

    save_strategy="no",
    remove_unused_columns=False,
    report_to="none",
    fp16=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

trainer.train()

In [ ]:
smooth = SmoothingFunction().method1

bleu_scores = []

for example in dataset:
    image = example["image"]
    reference = example["text"]

    inputs = processor(images=image, return_tensors="pt").to(device)

    # ✅ IMPROVED GENERATION
    output = model.generate(
        **inputs,
        num_beams=5,
        max_length=30,
        early_stopping=True
    )

    pred_caption = processor.decode(output[0], skip_special_tokens=True)

    reference_tokens = reference.split()
    pred_tokens = pred_caption.split()

    score = sentence_bleu(
        [reference_tokens],
        pred_tokens,
        smoothing_function=smooth
    )

    bleu_scores.append(score)

final_bleu = sum(bleu_scores) / len(bleu_scores)
print("Final BLEU Score:", final_bleu)

In [ ]:
model.save_pretrained("/kaggle/working/blip-final")
processor.save_pretrained("/kaggle/working/blip-final")